# OpenBB on nicogeek: quickstart

This notebook queries the self-hosted `openbb-api` (container `openbb-api`, port 6900) through the small helper in `obb.py`:

| Call | What it does |
|---|---|
| `obb.providers()` | data providers installed on the server |
| `obb.endpoints("text")` | find datasets (about 1,000 endpoints) |
| `obb.params("route")` | parameters of one endpoint |
| `obb.get("route", provider=..., **params)` | fetch data as a pandas DataFrame |

Keyless providers that work out of the box: **cboe, tmx, sec, federal_reserve, oecd, imf, ecb, us_treasury, usda, famafrench, finra, deribit, jodi**. On this server, use **cboe** for price history: nasdaq historical currently returns no data. FRED, BLS, EIA, CFTC and Congress need API keys configured on the server.

In [ ]:
import obb
import pandas as pd

pd.set_option("display.max_columns", 30)
print("Server:", obb.BASE_URL)
obb.providers()

## 1. Find a dataset

In [ ]:
obb.endpoints("historical")

In [ ]:
obb.params("equity/price/historical")

## 2. Stock prices

In [ ]:
start = (pd.Timestamp.today() - pd.DateOffset(months=6)).strftime("%Y-%m-%d")
aapl = obb.get("equity/price/historical", symbol="AAPL", start_date=start, provider="cboe")
aapl.tail()

In [ ]:
aapl.set_index("date")["close"].plot(title="AAPL close (cboe)", figsize=(10, 4), grid=True);

## 3. Indices: compare SPX and VIX

In [ ]:
spx = obb.get("index/price/historical", symbol="SPX", start_date=start, provider="cboe")
vix = obb.get("index/price/historical", symbol="VIX", start_date=start, provider="cboe")
both = pd.concat(
    {"SPX": spx.set_index("date")["close"], "VIX": vix.set_index("date")["close"]}, axis=1
)
both.plot(secondary_y="VIX", title="SPX vs VIX", figsize=(10, 4), grid=True);

cboe constituents cover the Cboe European indices (e.g. BUK100P = Cboe UK 100, BDE40P, BEP50P).

In [ ]:
obb.get("index/constituents", symbol="BUK100P", provider="cboe").head(20)

## 4. ETFs

In [ ]:
obb.get("etf/info", symbol="SPY", provider="nasdaq").T

In [ ]:
obb.get("etf/holdings", symbol="SPY", provider="nasdaq").head(15)

## 5. Federal Reserve regional data

Columns are discovered at runtime. These are wide tables, one column per series.

In [ ]:
obb.endpoints("federal_reserve/dallas")

In [ ]:
dallas = obb.get("federal_reserve/dallas/agricultural_survey")
dallas.tail()

## 6. Save to Excel or CSV

Files written here appear in the Jupyter file browser (left panel). Right-click a file and choose **Download** to open it on your Mac.

In [ ]:
with pd.ExcelWriter("openbb_export.xlsx") as writer:
    aapl.to_excel(writer, sheet_name="AAPL", index=False)
    both.to_excel(writer, sheet_name="SPX_VIX")
print("Saved openbb_export.xlsx")

## 7. Errors you may see

- **No data (HTTP 204):** the parameters matched nothing. The cell prints a message and returns an empty table.
- **`OBBError: HTTP 400: Missing credential 'fred_api_key'`:** add the key on the server (`user_settings.json` or an env var), then restart `openbb-api`.
- **`OBBError: HTTP 422 ...`:** a required parameter is missing or a value is invalid; the message names it.

In [ ]:
try:
    obb.get("economy/fred_series", symbol="GDP")
except obb.OBBError as error:
    print(error)